<div style="text-align: right">KBA-211008150613<TBD></div>
<div style="text-align: right">Rev 1.2</div>

<div style="text-align: center">Confidential – Qualcomm Technologies, Inc. and/or its affiliated companies – May Contain Trade Secrets</div>

<div style="text-align: center; font-weight: bold">MAY CONTAIN U.S. AND INTERNATIONAL EXPORT CONTROLLED INFORMATION</div>

**Takeaways:** Users will learn how to compile and execute the BERT base neural network model on Qualcomm® Cloud AI 100 (AIC) Inference Accelerator

**Before you start:** There are some commands (folder locations etc) that will need to be updated in this notebook based on your platform and installation location. Some commands might need sudo prefix to run properly.

**Last Verified Qualcomm Cloud AI 100 Platform SDK and Apps SDK Version:** 1.9.1.25 

# Qualcomm Cloud AI Getting started with BERT base


This notebook provides instructions for compiling and executing deep learning networks on the Qualcomm® Cloud AI 100 (AIC) Accelerator Card.  

The following topics are covered in this notebook:

**BERT base Example**
   > 1) Setup and Download the model \
   2) Model Preparation \
   3) Input data generation \
   3) Compilation \
   4) Inferencing \
   5) End-to-End Latency

# BERT Base Example

In this example we will download a BERT base model, compile the model in FP16 precision, then run the model on the AI 100 accelerator.

## Setup and Download the Model
For setup, install the required Python packages mentioned under requirements.txt and then download the pretrained bert-base model.

In [8]:
!cat bertbase_setup.sh

pip install -r requirements.txt

if [ -d generatedModels ]
then
  rm -rf generatedModels
fi

optimum-cli export onnx --model bert-base-cased --cache_dir model_files/cased --opset 11 --task question-answering generatedModels/ONNX/cased


In [9]:
!bash bertbase_setup.sh

2023-05-17 22:01:30.782263: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: :/opt/qti-aic/dev/lib/x86_64
2023-05-17 22:01:30.782296: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
Framework not specified. Using pt to export to ONNX.
Some weights of the model checkpoint at bert-base-cased were not used when initializing BertForQuestionAnswering: ['cls.predictions.transform.LayerNorm.weight', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.weight', 'cls.predictions.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.dense.bias']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained 

Using framework PyTorch: 1.11.0+cu102
Overriding 1 configuration item(s)
	- use_cache -> False
Post-processing the exported models...
Validating ONNX model generatedModels/ONNX/cased/model.onnx...
	-[���] ONNX model output names match reference model (end_logits, start_logits)
	- Validating ONNX Model output "start_logits":
		-[���] (2, 16) matches (2, 16)
		-[���] all values close (atol: 0.0001)
	- Validating ONNX Model output "end_logits":
		-[���] (2, 16) matches (2, 16)
		-[���] all values close (atol: 0.0001)
The ONNX export succeeded and the exported model was saved at: generatedModels/ONNX/cased


Here's the BERT base ONNX model.

In [10]:
!ls ./generatedModels/ONNX/cased

config.json  special_tokens_map.json  tokenizer_config.json
model.onnx   tokenizer.json	      vocab.txt


Here's the BERT configuration we'll be working with.

In [11]:
!cat ./generatedModels/ONNX/cased/config.json

{
  "_name_or_path": "bert-base-cased",
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.29.1",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 28996
}


## Model Preparation

The objective of the model preparator tool is to make sure model functionality and it performs well on AI 100 accelerator card.

Generally, trained models may have control flow operators, dynamic tensors, and other inference unfriendly operators or subgraphs. This tool applies precompile optimization and cleans the model.

The parameters required by the tool are configured in a configuration file in YAML format (**bertbase.yaml**) and the tool parses this configuration file to execute.

In [12]:
!cat bertbase.yaml

##############################################################################
#
# Copyright (c) 2023 Qualcomm Technologies, Inc.
# All Rights Reserved.
# Confidential and Proprietary - Qualcomm Technologies, Inc.
#
# All data and information contained in or disclosed by this document are
# confidential and proprietary information of Qualcomm Technologies, Inc., and
# all rights therein are expressly reserved. By accepting this material, the
# recipient agrees that this material and the information contained therein
# are held in confidence and in trust and will not be used, copied, reproduced
# in whole or in part, nor its contents revealed in any manner to others
# without the express written permission of Qualcomm Technologies, Inc.
#
##############################################################################

# Official Model Location: https://huggingface.co/models

MODEL:
    INFO:
        DESCRIPTION: "HuggingFace BERT base cased Model"
        MODEL_TYPE: TRANSFORMERS
       

In [14]:
!cat model-preparator.sh

###############################################################################
# Copyright (c) 2023 Qualcomm Technologies, Inc.
# All Rights Reserved.
# Confidential and Proprietary - Qualcomm Technologies, Inc.

# All data and information contained in or disclosed by this document are
# confidential and proprietary information of Qualcomm Technologies, Inc., and
# all rights therein are expressly reserved. By accepting this material, the
# recipient agrees that this material and the information contained therein
# are held in confidence and in trust and will not be used, copied, reproduced
# in whole or in part, nor its contents revealed in any manner to others
# without the express written permission of Qualcomm Technologies, Inc.
###############################################################################

#!/bin/bash
#complete modelpath
model_output_path=./generatedModels/ONNX/cased/prepmodel.onnx

#preparing the model as per AIC
prepare_model(){
	if [ -d WORKSPACE ]
	then
		rm

In [15]:
!bash model-preparator.sh

2023-05-17 22:12:47.883 | WARNING  | __main__:prepare_work_dir:35 - Directory WORKSPACE will be deleted if already exists. Take backup before execution.
2023-05-17 22:12:47.888 | INFO     | __main__:main:78 - QAic Model Preparator Tool
2023-05-17 22:12:48.264 | INFO     | qaic_pytools.qmodel.backend.onnx.model:native_checker:638 - Graph checker passed successfully.
2023-05-17 22:12:48.594 | INFO     | qaic_pytools.qmodel.preparator.preparator:__prepare_onnx:215 - Native Checker on the input model is successful
2023-05-17 22:12:48.594 | INFO     | qaic_pytools.qmodel.preparator.preparator:__prepare_onnx:219 - Native Checker on the model is successful
2023-05-17 22:12:49.530 | INFO     | qaic_pytools.qmodel.preparator.preparator:__prepare_onnx:231 - Internal Checker (Ort) on the model is successful
2023-05-17 22:12:49.712 | WARNING  | qaic_pytools.qmodel.backend.onnx.onnx_model:native_shape_inference:613 - Symbolic shape inference failed. Exception: Incomplete symbolic shape inference. R

��������������������������������������������������������������������� model_preparator_aic100 Summary ������������������������������������������������������������������������
��� IR Version: 6                                                                ���
��� Opset Version: 11 , 11 com.qti.aisw.onnx,                                    ���
��� Producer Name:                                                               ���
��� Doc:                                                                         ���
��� Total Count of Ops: 947                                                      ���
��� QModel Tool Version: 0.0.1                                                   ���
��� Total Model Parameters: 107,721,743.0                                        ���
���                                                                              ���
��� All Ops: {'Shape': 97, 'Gather': 100, 'Add': 161, 'Unsqueeze': 97, 'Slice':  ���
��� 1, 'ReduceMean': 50, 'Sub': 25, 'Pow': 25, 'Sqrt': 25, '

In [16]:
!ls ./generatedModels/ONNX/cased

config.json  prepmodel.onnx	      tokenizer.json	     vocab.txt
model.onnx   special_tokens_map.json  tokenizer_config.json


## Modification
Modify the onnx file to handle `constants > FP16_Max and < FP16_Min`

In [72]:
import onnx
from onnx import numpy_helper
import numpy as np

model_card = 'bert-base-cased'
gen_models_path = f"generatedModels/ONNX/cased"
model_base_name = model_card

def fix_onnx_fp16(
    gen_models_path: str,
    model_base_name: str,
) -> str:
    finfo = np.finfo(np.float16)
    fp16_max = finfo.max
    fp16_min = finfo.min

    model = onnx.load(f"generatedModels/ONNX/cased/model.onnx")
    fp16_fix = False
    for tensor in onnx.external_data_helper._get_all_tensors(model):
        nptensor = numpy_helper.to_array(tensor, gen_models_path)
        if nptensor.dtype == np.float32 and (
            np.any(nptensor > fp16_max) or np.any(nptensor < fp16_min)
        ):
            # print(f'tensor value : {nptensor} above {fp16_max} or below {fp16_min}')
            nptensor = np.clip(nptensor, fp16_min, fp16_max)
            new_tensor = numpy_helper.from_array(nptensor, tensor.name)
            tensor.CopyFrom(new_tensor)
            fp16_fix = True
            
    if fp16_fix:
        # Save FP16 model
        print("Found constants out of FP16 range, clipped to FP16 range")
        model_base_name += "_fix_outofrange_fp16"
        onnx.save(model, f=f"{gen_models_path}/{model_base_name}.onnx")
        print(f"Saving modified onnx file at {gen_models_path}/{model_base_name}.onnx")
    return model_base_name

fp16_model_name = fix_onnx_fp16(gen_models_path=gen_models_path, model_base_name=model_base_name)

Found constants out of FP16 range, clipped to FP16 range
Saving modified onnx file at generatedModels/ONNX/cased/bert-base-cased_fix_outofrange_fp16.onnx


In [73]:
!ls ./generatedModels/ONNX/cased

bert-base-cased_fix_outofrange_fp16.onnx  special_tokens_map.json
config.json				  tokenizer.json
model.onnx				  tokenizer_config.json
prepmodel.onnx				  vocab.txt


## Input data generation
In this step we generate random inputs for BERT base model using **input-data-generation.sh** script, which will be used during inference request.

In [17]:
!cat input-data-generation.sh

###############################################################################
# Copyright (c) 2019-2020 Qualcomm Technologies, Inc.
# All Rights Reserved.
# Confidential and Proprietary - Qualcomm Technologies, Inc.

# All data and information contained in or disclosed by this document are
# confidential and proprietary information of Qualcomm Technologies, Inc., and
# all rights therein are expressly reserved. By accepting this material, the
# recipient agrees that this material and the information contained therein
# are held in confidence and in trust and will not be used, copied, reproduced
# in whole or in part, nor its contents revealed in any manner to others
# without the express written permission of Qualcomm Technologies, Inc.
###############################################################################

#!/bin/bash

#Input data generation
python -W ignore generateInputs.py --seq_len 128 --batch_size=1


In [19]:
!bash input-data-generation.sh

Inputs are generated under **inputFiles** folder.

In [20]:
!ls ./inputFiles

input_ids_sl128_bs1.raw  input_mask_sl128_bs1.raw  segment_ids_sl128_bs1.raw


## Compile the Network
In this step we generate AIC100 network binaries from the ONNX model using the `qaic-exec` compiler.  The binaries are generated in the `./compiler_output_FP16` folder. The network is compiled with the following optimizations:
* cores - Network will execute on 2 NSP compute cores
* ols - Overlap split factor (increases splitting of network operations) is set to 1
* mos - Maximum output channel split (optimization of on-chip memory usage) is set to 1

In [22]:
!rm -rf ./compiler_output_FP16

!/opt/qti-aic/exec/qaic-exec \
    -model=./generatedModels/ONNX/cased/prepmodel.onnx \
    -aic-hw-version=2.0 \
    -aic-hw \
    -convert-to-fp16 \
    -aic-num-cores=1 \
    -onnx-define-symbol=batch_size,1  \
    -stats-batchsize=1 \
    -onnx-define-symbol=seq_len,128 \
    -multicast-weights \
    -aic-binary-dir=./compiler_output_FP16 \
    -compile-only

Reading ONNX Model from ./generatedModels/ONNX/cased/prepmodel.onnx
loading compiler from: /opt/qti-aic/dev/lib/x86_64/libQAicCompiler.so
Compile started ............... 
Compiling model with FP16 precision.
Generated binary is present at ./compiler_output_FP16


## Inferencing
Next we run the compiled network binaries on AIC100 hardware with the `qaic-runner` tool to check the throughput.  We use one of the input sequences generated during the 'Generate Input Data' step above.

Inferencing runs on `device id 0 by default`.  To specify a different card, add the `--aic-device-id` option.  For example: `--aic-device-id 1` to run on device 1.  Run the `/opt/qti-aic-tools/qaic-util -q` tool to list the device IDs.

In [65]:
!/opt/qti-aic/exec/qaic-runner \
    --test-data ./compiler_output_FP16 \
    --aic-num-of-activations 1 \
    --set-size 10 \
    --num-iter 1000 \
    --input-file inputFiles/input_ids_sl128_bs1.raw \
    --input-file inputFiles/input_mask_sl128_bs1.raw \
    --input-file inputFiles/segment_ids_sl128_bs1.raw \
    --aic-device-id 0

loading /opt/qti-aic/dev/lib/x86_64/libQAic.so
Input file: inputFiles/input_ids_sl128_bs1.raw
Input file: inputFiles/input_mask_sl128_bs1.raw
Input file: inputFiles/segment_ids_sl128_bs1.raw
 ---- Stats ----
InferenceCnt 1000 TotalDuration 7826921us BatchSize 1 Inf/Sec 127.764


## End-to-End Latency

Latency statistics are available from the `qaic-runner` tool with the following options:

```
Usage: qaic-runner [options]
  -S, --set-size <i>                 Set Size for inference loop execution, default:10, min:1
  -l, --live-reporting               Enable Live reporting periodic at 1 sec interval, default off
  -s, --stats                        Enable Live Profiling Stats reporting periodically at 1 sec interval
```

End-to-end latency is calculated as:<br>
preProcTime + execTotalTime + postProcTime

Set size of 1 `--set-size 1` is recommended for latency measurement.  

<img src="./images/latency_e2e.png" alt="Latency Breakdown" />

In [62]:
# Collect latency stats output for <max_run> runs

max_run = 3
results = [None] * max_run

for idx in range(0, max_run):
    print('Run {}'.format(idx))
    results[idx] = !/opt/qti-aic/exec/qaic-runner \
        --test-data ./compiler_output_FP16 \
        --aic-num-of-activations 1 \
        --num-iter 1000 \
        --input-file inputFiles/input_ids_sl128_bs1.raw \
        --input-file inputFiles/input_mask_sl128_bs1.raw \
        --input-file inputFiles/segment_ids_sl128_bs1.raw \
        --aic-device-id 0 \
        --set-size 1 \
        --live-reporting 1000 \
        --stats

Run 0
Run 1
Run 2


Define a helper class QAicLatencyStats to parse the latency stats.

In [63]:
import re
import pandas as pd

class QAicLatencyStats:
    def __init__(self):
        pass
    
    def load(self, results):
        measurement_map = {'preProcessingLatencyStats': 'pre_proc_us',
                           'execTotalTimeStats': 'exec_total_us',
                           'execKernelToCompletStats': 'exec_kernel_to_complete_us',
                           'postProcessingLatencyStats': 'post_proc_us'}

        columns = ['activation', 'num_samples', 'timestamp_ms', 'num_inf_completed', 'rate_inf_sec']
        data = {}
        for column in columns:
            data[column] = []

        # Add measurement columns
        for key,value in measurement_map.items():
            data[value] = []
        
        for result in results:
            match = re.match('Activation\[([0-9]+)\]', result)
            if match:
                activation = int(match.group(1))
                #print('activation {}'.format(activation))
                data['activation'].append(activation)

            match = re.match('NumSamples\:\s+([0-9]+)', result)
            if match:
                num_samples = int(match.group(1))
                #print('num_samples {}'.format(num_samples))
                data['num_samples'].append(num_samples)

            match = re.match('([\w]+) \(us\).+avg\:\s+([0-9\.]+)', result)
            if match:
                measurement = match.group(1)
                if measurement in measurement_map.keys():
                    measurement = measurement_map[measurement]
                    latency = float(match.group(2))
                    #print('{} latency {}'.format(measurement, latency))
                    data[measurement].append(latency)

            #timestamp:    9000ms numInfCompleted:      881 rate:   97.889 Inf/Sec
            match = re.match('timestamp\:\s+([0-9]+)ms\s+numInfCompleted\:\s+([0-9]+)\s+rate\:\s+([0-9\.]+) Inf\/Sec', result)
            if match:
                timestamp = int(match.group(1))
                num_inf_completed = int(match.group(2))
                rate_inf_sec = float(match.group(3))
                #print('timestamp {}'.format(timestamp))
                data['timestamp_ms'].append(timestamp)
                data['num_inf_completed'].append(num_inf_completed)
                data['rate_inf_sec'].append(rate_inf_sec)
                                
        df = pd.DataFrame.from_dict(data)
        df['e2e_total_latency_us'] = df['pre_proc_us'] + df['exec_total_us'] + df['post_proc_us']
        return df

Parse and show the average latency results for each activation.

In [64]:
latency_stats = QAicLatencyStats()

avg = 0
df = pd.DataFrame()
for idx in range(0, max_run):
    print('Results {}'.format(idx))
    run_df = latency_stats.load(results[idx])
    avg += run_df['e2e_total_latency_us'].mean()
    run_df['run'] = idx
    df = pd.concat([run_df, df])

print('Average e2e latency: {:0.2f} us'.format(avg / max_run))

#df = df.reindex(columns=['run', 'activation', 'e2e_total_latency_us', 'rate_inf_sec', 'num_samples', 'timestamp_ms', 'num_inf_completed', 'pre_proc_us', 'exec_total_us', 'exec_kernel_to_complete_us', 'post_proc_us'])

#display(df.sort_values(by=['run', 'activation', 'timestamp_ms']))

Results 0
Results 1
Results 2
Average e2e latency: 7905.84 us


<br>
<div style="text-align: center">Confidential – Qualcomm Technologies, Inc. and/or its affiliated companies – May Contain Trade Secrets</div>

<div style="text-align: center; font-weight: bold">MAY CONTAIN U.S. AND INTERNATIONAL EXPORT CONTROLLED INFORMATION</div>